## Async Random Sentences to S3

Create a batch of random sentences, store them in S3 through LAILA, persist a `{nickname: global_id}` manifest, and await the batched `GroupFuture` directly.


In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
import random
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/ubuntu/laila")
PYTHON_ROOT = PROJECT_ROOT.parent
if str(PYTHON_ROOT) not in sys.path:
    sys.path.append(str(PYTHON_ROOT))

import laila
laila.read_args(PROJECT_ROOT / "vault" / "dev_secrets.toml")
from laila.pool import S3Pool


In [8]:
POOL_NICKNAME = "my_sentences_pool"
MANIFEST_NICKNAME = "my_sentences_manifest"
NUM_SENTENCES = 25

subjects = ["The robot", "A scientist", "My neighbor", "The artist", "An astronaut", "The teacher"]
verbs = ["builds", "discovers", "writes", "studies", "launches", "observes"]
objects = ["a tiny engine", "a strange pattern", "a new theorem", "the old telescope", "a paper airplane", "a hidden garden"]
adverbs = ["carefully", "enthusiastically", "quietly", "overnight", "without hesitation", "before sunrise"]


def build_s3_pool() -> S3Pool:
    required = ["AWS_BUCKET_NAME", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_REGION"]
    missing = [name for name in required if getattr(laila.args, name, None) is None]
    if missing:
        raise RuntimeError(f"Missing laila.args values: {', '.join(missing)}")

    return S3Pool(
        bucket_name=laila.args.AWS_BUCKET_NAME,
        access_key_id=laila.args.AWS_ACCESS_KEY_ID,
        secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
        region_name=laila.args.AWS_REGION,
        nickname=POOL_NICKNAME,
    )


def make_random_sentence() -> str:
    return " ".join(
        [
            random.choice(subjects),
            random.choice(verbs),
            random.choice(objects),
            random.choice(adverbs) + ".",
        ]
    )


In [9]:
s3_pool = build_s3_pool()
laila.add_pool(s3_pool, pool_nickname=POOL_NICKNAME)

sentence_entries = []
sentence_manifest = {}
for i in range(NUM_SENTENCES):
    sentence_name = f"random_sentence_{i}"
    sentence = make_random_sentence()
    entry = laila.constant(data=sentence, nickname=sentence_name)
    sentence_entries.append(entry)
    sentence_manifest[sentence_name] = entry.global_id

manifest_entry = laila.constant(data=sentence_manifest, nickname=MANIFEST_NICKNAME)

list(sentence_manifest.items())[:3]

[('random_sentence_0',
  'LAILA:ENTRY:GLOBAL_ID:050fd403-8fdd-53fb-830c-c50b713ba8aa'),
 ('random_sentence_1',
  'LAILA:ENTRY:GLOBAL_ID:e1117353-174e-5130-b86d-3fc604ad0a7d'),
 ('random_sentence_2',
  'LAILA:ENTRY:GLOBAL_ID:282b3733-8e56-5497-b39e-23e7d8812a67')]

In [10]:
upload_futures = laila.memorize(sentence_entries, pool_nickname=POOL_NICKNAME)
manifest_future = laila.memorize(manifest_entry, pool_nickname=POOL_NICKNAME)

print("Initial sentence upload status:", upload_futures.status)
await upload_futures
await manifest_future

print("\nFinal sentence upload status:", upload_futures.status)
print("Uploaded sentence futures:", len(upload_futures))
print("Manifest global id:", manifest_entry.global_id)

Initial sentence upload status: {'total': 25.0, 'percentages': {'finished': 0.0, 'running': 0.0, 'not_started': 100.0, 'error': 0.0, 'cancelled': 0.0}}

Final sentence upload status: {'total': 25.0, 'percentages': {'finished': 100.0, 'running': 0.0, 'not_started': 0.0, 'error': 0.0, 'cancelled': 0.0}}
Uploaded sentence futures: 25
Manifest global id: LAILA:ENTRY:GLOBAL_ID:67bab0b6-19b3-5988-a1f0-2fee9e01af43


In [ ]:
manifest_future = laila.remember(manifest_entry.global_id, pool_nickname=POOL_NICKNAME)
remembered_manifest_entry = await manifest_future
remembered_manifest = dict(remembered_manifest_entry.data)

remember_names = list(remembered_manifest.keys())
remember_futures = laila.remember(
    list(remembered_manifest.values()),
    pool_nickname=POOL_NICKNAME,
)

remembered_entries = await remember_futures
remembered_sentences = {
    name: entry.data
    for name, entry in zip(remember_names, remembered_entries)
}

list(remembered_sentences.items())[:3]

[('random_sentence_0', 'The artist builds a paper airplane quietly.'),
 ('random_sentence_1', 'An astronaut observes a paper airplane carefully.'),
 ('random_sentence_2',
  'My neighbor studies a paper airplane enthusiastically.')]